In [1]:
import pandas as pd

world_heritage_sites = pd.read_csv("../data/raw/whc001.csv")

In [2]:

# Keep English-language fields because the project uses English Wikipedia
# Drop image- and video-related fields
# Drop Date end because it is 100% missing
# Drop text-based fields for the first model, except Name EN for interpretability
# Drop Danger data, Danger = True is only about 5% of the data and then there is a portion that's added in 2026 making it even more sparse
# Drop identifiers, except ID, which is used as the primary key for merging
# Drop Region Code because its information is already captured by Region
# Drop Coordinates because geographic location is already represented by other fields
print(world_heritage_sites.columns.tolist())
columns_to_keep = [
    'Name EN',
    'Date inscribed',
    'Category',
    'ISO Codes',
    'Region',
    "States Names",
    'ID',
    "Coordinates",
    "Transboundary",
    "Danger",
    "Components Count"
]

world_heritage_sites = world_heritage_sites[columns_to_keep]



# 5 in 2026 with already sparse data on danger = true im going to ommit


['Name EN', 'Name FR', 'Name ES', 'Name RU', 'Name AR', 'Name ZH', 'Short Description EN', 'Short Description FR', 'Short Description ES', 'Short Description RU', 'Short Description AR', 'Short Description ZH', 'Description EN', 'Justification EN', 'Date inscribed', 'Secondary dates', 'Danger', 'Date end', 'Danger list', 'Area hectares', 'Cultural Criteria', 'Natural Criteria', 'Criteria', 'Category', 'Category ID', 'States Names', 'ISO Codes', 'Region', 'Region Code', 'Transboundary', 'Main Image', 'Main Image Author', 'Main Image Copyright', 'Main Image Caption EN', 'Main Image Caption FR', 'Main Image Caption ES', 'Main Image Caption RU', 'Main Image Caption AR', 'Main Image Caption ZH', 'Images', 'UUID', 'ID', 'Coordinates', 'Main Video', 'Main Video Author', 'Main Video Caption EN', 'Main Video Caption FR', 'Main Video Caption ES', 'Main Video Caption RU', 'Main Video Caption AR', 'Main Video Caption ZH', 'videos_urls', 'Components', 'Components Count']


In [4]:
import numpy as np
import country_converter as coco

# 1. Derive continent from ISO country codes


def get_continent(iso_codes):
    if pd.isna(iso_codes):
        return np.nan

    codes = [
        code.strip()
        for code in str(iso_codes).split(",")
        if code.strip()
    ]

    if not codes:
        return np.nan

    results = coco.convert(
        names=codes,
        to="Continent",
        not_found=None
    )

    if isinstance(results, str):
        results = [results]

    continents = {
        c for c in results
        if isinstance(c, str)
        and c.lower() not in ("not found", "not_found", "none", "")
    }

    if len(continents) == 1:
        return next(iter(continents))

    if len(continents) > 1:
        return "Multiple"

    return np.nan


world_heritage_sites["Continent"] = (
    world_heritage_sites["ISO Codes"].apply(get_continent)
)


# 2. Inspect results


print("Continent counts:")
print(world_heritage_sites["Continent"].value_counts(dropna=False))

print("\nSites with missing continents:")
print(
    world_heritage_sites.loc[
        world_heritage_sites["Continent"].isna(),
        ["Name EN", "States Names", "ISO Codes", "Continent"]
    ].to_string(index=False)
)

print("\nSites classified as Multiple:")
print(
    world_heritage_sites.loc[
        world_heritage_sites["Continent"].eq("Multiple"),
        ["Name EN", "States Names", "ISO Codes", "Continent"]
    ].to_string(index=False)
)
# 3. Apply  manual corrections

world_heritage_sites.loc[
    world_heritage_sites["States Names"].eq("Namibia"),
    "Continent"
] = "Africa"

# Landscapes of Dauria and Uvs Nuur Basin are in Asia
asian_sites = [
    "Landscapes of Dauria",
    "Uvs Nuur Basin"
]

world_heritage_sites.loc[
    world_heritage_sites["Name EN"].isin(asian_sites),
     "Continent"
] = "Asia"

Continent counts:
Continent
Europe      495
Asia        383
America     200
Africa      156
Oceania      32
Multiple      4
NaN           3
Name: count, dtype: int64

Sites with missing continents:
                            Name EN                        States Names ISO Codes Continent
                     Namib Sand Sea                             Namibia       NaN       NaN
         Twyfelfontein or /Ui-//aes                             Namibia       NaN       NaN
Old City of Jerusalem and its Walls Jerusalem (Site proposed by Jordan)       NaN       NaN

Sites classified as Multiple:
                                                                                   Name EN                                                                                  States Names                  ISO Codes Continent
                                                                      Landscapes of Dauria                                                                   Mongolia,Russian Federat

In [5]:

import numpy as np
import pandas as pd

# 1. Load and clean geographic datasets

cities = pd.read_csv("../data/raw/world_cities.csv")
airports = pd.read_csv("../data/raw/airports.csv")

EARTH_RADIUS_KM = 6371.0
BATCH_SIZE = 25

# Clean cities
cities = cities.loc[
    pd.to_numeric(cities["pop"], errors="coerce").notna()
].copy()

for col in ["lat", "lng", "pop"]:
    cities[col] = pd.to_numeric(cities[col], errors="coerce")

cities = cities.loc[
    cities["pop"].ge(0)
    & cities["lat"].between(-90, 90)
    & cities["lng"].between(-180, 180)
].dropna(subset=["lat", "lng", "pop"]).copy()

# Clean airports
for col in ["latitude_deg", "longitude_deg"]:
    airports[col] = pd.to_numeric(airports[col], errors="coerce")

airports = airports.loc[
    airports["latitude_deg"].between(-90, 90)
    & airports["longitude_deg"].between(-180, 180)
].dropna(subset=["latitude_deg", "longitude_deg"]).copy()

# Large airports
large_scheduled_airports = airports.loc[
    airports["type"].eq("large_airport")
    & airports["scheduled_service"]
        .astype(str).str.strip().str.lower()
        .isin(["yes", "true", "1"])
].copy()


# 2. Extract UNESCO coordinates

coords = (
    world_heritage_sites["Coordinates"]
    .astype("string")
    .str.split(",", expand=True)
    .apply(pd.to_numeric, errors="coerce")
    .to_numpy(dtype=float)
)

valid_sites = (
    np.isfinite(coords).all(axis=1)
    & (coords[:, 0] >= -90)
    & (coords[:, 0] <= 90)
    & (coords[:, 1] >= -180)
    & (coords[:, 1] <= 180)
)

valid_positions = np.flatnonzero(valid_sites)
site_rad = np.radians(coords[valid_sites])
n_sites = len(world_heritage_sites)

# Prepare coordinate arrays
city_lat = np.radians(cities["lat"].to_numpy(dtype=float))
city_lon = np.radians(cities["lng"].to_numpy(dtype=float))
city_pop = cities["pop"].to_numpy(dtype=float)

airport_lat = np.radians(
    large_scheduled_airports["latitude_deg"].to_numpy(dtype=float)
)
airport_lon = np.radians(
    large_scheduled_airports["longitude_deg"].to_numpy(dtype=float)
)


# 3. Haversine distance calculation

def distance_matrix(site_rad, target_lat, target_lon):
    """Return distances in km from each site to each target."""
    lat1 = site_rad[:, 0, None]
    lon1 = site_rad[:, 1, None]

    dlat = target_lat[None, :] - lat1
    dlon = target_lon[None, :] - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(target_lat[None, :])
        * np.sin(dlon / 2) ** 2
    )

    return (
        2 * EARTH_RADIUS_KM
        * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    )


# 4. Initialize the eight metrics


metric_names = [
    "city_population_within_50km",
    "city_population_within_100km",
    "nearest_city_distance_km",
    "nearest_city_100k_distance_km",
    "nearest_city_500k_distance_km",
    "nearest_large_scheduled_airport_distance_km",
    "large_scheduled_airports_within_100km"
]

metrics = {
    name: np.full(n_sites, np.nan)
    for name in metric_names
}

city_pop_50k = city_pop >= 100_000
city_pop_100k = city_pop >= 500_000


# 5. Calculate metrics in batches

for start in range(0, len(site_rad), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(site_rad))
    batch = site_rad[start:end]
    positions = valid_positions[start:end]

    # City distances
    city_dist = distance_matrix(batch, city_lat, city_lon)

    metrics["city_population_within_50km"][positions] = (
        (city_dist <= 50) * city_pop[None, :]
    ).sum(axis=1)

    metrics["city_population_within_100km"][positions] = (
        (city_dist <= 100) * city_pop[None, :]
    ).sum(axis=1)

    metrics["nearest_city_distance_km"][positions] = (
        city_dist.min(axis=1)
    )

    # Nearest cities above population thresholds
    for threshold, mask, name in [
        (50_000, city_pop_50k, "nearest_city_100k_distance_km"),
        (100_000, city_pop_100k, "nearest_city_500k_distance_km")
    ]:
        if mask.any():
            metrics[name][positions] = city_dist[:, mask].min(axis=1)

    # Large airport distances
    if len(airport_lat):
        airport_dist = distance_matrix(batch, airport_lat, airport_lon)

        metrics["nearest_large_scheduled_airport_distance_km"][positions] = (
            airport_dist.min(axis=1)
        )

        metrics["large_scheduled_airports_within_100km"][positions] = (
            airport_dist <= 100
        ).sum(axis=1)

# Add metrics to the UNESCO dataset
for name, values in metrics.items():
    world_heritage_sites[name] = values

# 6. Inspect results

print("Number of UNESCO properties:", n_sites)
print("Properties with valid coordinates:", len(valid_positions))

print("\nMetric summaries:")
print(world_heritage_sites[metric_names].describe().round(2))

print("\nMissing values:")
print(world_heritage_sites[metric_names].isna().sum())

# Optional: save the enriched dataset
world_heritage_sites.to_csv(
    "../data/processed/heritage_remoteness_metrics.csv",
    index=False
)


Number of UNESCO properties: 1273
Properties with valid coordinates: 1234

Metric summaries:
       city_population_within_50km  city_population_within_100km  \
count                      1234.00                       1234.00   
mean                     899675.57                    1833988.62   
std                     2351378.26                    3211330.00   
min                           0.00                          0.00   
25%                           0.00                     153113.00   
50%                      118984.50                     676071.00   
75%                      686750.25                    1996012.12   
max                    18162549.50                   28873765.50   

       nearest_city_distance_km  nearest_city_100k_distance_km  \
count                   1234.00                        1234.00   
mean                      55.52                         127.34   
std                      180.33                         351.04   
min                        0.0

In [6]:
world_heritage_sites = world_heritage_sites[
    world_heritage_sites["Date inscribed"] <= 2025
]

world_heritage_sites["inscription_year"] = pd.to_numeric(
    world_heritage_sites["Date inscribed"],
    errors="coerce"
)

world_heritage_sites["years_since_inscription"] = (
    2026 - world_heritage_sites["inscription_year"]
)




In [7]:
# removing columns :
world_heritage_sites = world_heritage_sites.drop(
    columns=["ISO Codes", "States Names", "Transboundary","Coordinates","inscription_year","Date inscribed"]
 )

In [8]:

wikipedia_articles = pd.read_csv("../data/raw/wikipedia_views.csv")

# Filter for January 2026
january_2026_wikipedia_views = wikipedia_articles[
    wikipedia_articles["month"] == 202601
].copy()

# Sum pageviews across languages for each UNESCO site
january_2026_wikipedia_views = (
    january_2026_wikipedia_views
    .groupby("ID", as_index=False)["views"]
    .sum()
    .rename(columns={"views": "total_views"})
)

# Merge pageviews into the World Heritage dataset
final_heritage_jan_2026 = world_heritage_sites.merge(
    january_2026_wikipedia_views,
    on="ID",
    how="left"
)

# Save the final dataset
final_heritage_jan_2026.to_csv(
    "../data/processed/final_heritage_jan_2026.csv",
    index=False
)

# Check the results
print(final_heritage_jan_2026.head())
print(f"Total UNESCO sites: {len(final_heritage_jan_2026)}")
print(f"Sites with pageview data: {final_heritage_jan_2026['total_views'].notna().sum()}")
print(f"Sites without pageview data: {final_heritage_jan_2026['total_views'].isna().sum()}")

                                             Name EN  Category  \
0                        Murujuga Cultural Landscape  Cultural   
1  Megaliths of Carnac and of the shores of Morbihan  Cultural   
2                            Minoan Palatial Centres  Cultural   
3               Maratha Military Landscapes of India  Cultural   
4  Forest Research Institute Malaysia Forest Park...  Cultural   

                     Region    ID  Danger  Components Count Continent  \
0      Asia and the Pacific  1709   False                 1   Oceania   
1  Europe and North America  1725   False                 4    Europe   
2  Europe and North America  1733   False                 6    Europe   
3      Asia and the Pacific  1739   False                12      Asia   
4      Asia and the Pacific  1734   False                 1      Asia   

   city_population_within_50km  city_population_within_100km  \
0                      16636.0                       27917.5   
1                      71532.0      